# 12 Bark Detector

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np

from woofdata.loader import load_sample
from woofdata.path import data_path, load_data_dir_from_env_file
from woofdata.tasks import read_manifest, task_rows
from woofdecoder.preprocess.bark_detector.events import frames_to_events
from woofdecoder.preprocess.bark_detector.model import load_bark_detector
from woofdecoder.preprocess.bark_detector.predict import detect_sed_probs
from woofdecoder.preprocess.bark_detector.pretrained_sed import PRETRAINED_SED_DIR
from woofdecoder.preprocess.bark_detector.timing import FRAME_S
from woofdecoder.preprocess.media import load_audio_16k
from woofdecoder.sed_classes import SED_CLASSES
from wooftrainer.eval.metrics import event_scores, frame_scores

TASK = "bark_detector"
CACHE_PATH = data_path("cache/bark_detector/playpant_val_zero_shot.npz")

## Pretrained SED Check 

In [2]:
load_data_dir_from_env_file()

print(f"{PRETRAINED_SED_DIR}")
print(f"Exists: {PRETRAINED_SED_DIR.is_dir()}")

C:\Users\kimy8\Desktop\Deep Learning\dog-vocal-interpreter\third_party\PretrainedSED
Exists: True


## Dataset Check

In [3]:
manifest = read_manifest()
for source in ["playpant", "audioset_strong_eval", "esc50"]:
    counts = manifest.loc[manifest["source"] == source, "split"].value_counts()
    print(f"{source:<22} {dict(counts)}")

audioset_rows = manifest[manifest["source"] == "audioset_strong_eval"]
print(audioset_rows["original_label"].value_counts().head(15).to_string())

## Load Bark Detector

### Basic Check on Test Video

In [4]:
detector = load_bark_detector()

audio = load_audio_16k(data_path("test/media_test.mp4"))
start = time.perf_counter()
sed_probs = detect_sed_probs(detector, audio)
seconds_taken = time.perf_counter() - start
print(f"Output shape: {sed_probs.shape} (frames x {SED_CLASSES})")
print(f"{len(audio) / 16_000:.1f} s of audio took {seconds_taken:.2f} s")

dilation_list_t_dim: 
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
MN(
  (features): Sequential(
    (0): ConvNormActivation(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): ConvNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): ConvNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, bias=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Con

c:\Users\kimy8\Desktop\Deep Learning\dog-vocal-interpreter\.venv\Lib\site-packages\torchvision\ops\misc.py:121: UserWarning: Don't use ConvNormActivation directly, please use Conv2dNormActivation and Conv3dNormActivation instead.
  warnings.warn(
100%|██████████| 14.8M/14.8M [00:00<00:00, 24.7MB/s]
C:\Users\kimy8\Desktop\Deep Learning\dog-vocal-interpreter\third_party\PretrainedSED\models\frame_passt\preprocess.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


3.83M parameters, 447 classes
bark         <- ['Bark', 'Yip', 'Bow-wow']
growl        <- ['Growling']
whine        <- ['Whimper (dog)']
howl         <- ['Howl']
pant         <- ['Pant (dog)']
other_vocal  <- ['Dog', 'Canidae, wild dogs, wolves']
Loading audio from media_test.mp4
Read 240,640 samples, 2 channels, 44,100 Hz
Converted to mono 16,000 Hz: 87,307, samples = 5.46 s
Output shape: (137, 6) (frames x ['bark', 'growl', 'whine', 'howl', 'pant', 'other_vocal'])
5.5 s of audio took 0.06 s


Has 17 layers with indicies of 0 to 16 in which does the following:
- (0) A stem conv from 1 to 16 channels, w/ stride of 2 on both axes
- (1)-(15) Inverted residual blocks
- (16) 1 by 1 conv that widens 160 to 960 channels

That 960-dim per-frame vector is a general-purpose audio embedding trained on AudioSet.

### Show Events above 0.5


In [5]:
dog_prob = sed_probs.max(axis=1)
for start_s, end_s in frames_to_events(dog_prob >= 0.5):
    print(f"{start_s:6.2f} - {end_s:6.2f} s")

times = np.arange(len(sed_probs)) * FRAME_S
plt.figure(figsize=(14,3))
for column, sed_class in enumerate(SED_CLASSES):
    plt.plot(times, sed_probs[:, column], label=sed_class, linewidth=1)
plt.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("time (s)")
plt.ylabel("probability")
plt.title("Bark detector on test video")
plt.legend(ncol=6)
plt.show()

"Mel" comes from "melody". The mel scale is a perceptual pitch scale. The scale is built so that equal steps in mels sound like equal steps in pitch to a human.


Mel is modelled on human hearing. The model uses it because it is compact (257 --> 128 bins of info), matches natural sounds, and it is the convention.

## Baseline on playpant

In [6]:
val_rows = task_rows(TASK, "val", sources = ["playpant"])
print(f"{len(val_rows)} windows of 10 s\n")

all_probs, all_truth = [], []
for number, (_,row) in enumerate(val_rows.iterrows(), start =1):
    sample = load_sample(TASK, row)
    sed_probs = detect_sed_probs(detector, sample.audio)[: len(sample.targets)]
    all_probs.append(sed_probs)
    all_truth.append(sample.targets)
    if number % 100 == 0 or number == len(val_rows):
        print(f"{number}/{len(val_rows)} done")
        
val_probs = np.stack(all_probs)
val_targets = np.stack(all_truth)

CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
np.savez(CACHE_PATH, probs=val_probs, targets=val_targets, clip_ids=val_rows["clip_id"].to_numpy())
print(f"\nSaved to {CACHE_PATH}")

true_dog = val_targets.any(axis=2)
print(f"Frames with a dog sound: {true_dog.mean():.1%}")
for column, sed_class in enumerate(SED_CLASSES):
    print(f"{sed_class:<12} {val_targets[:, :, column].mean():.2%} of frames")
    
dog_prob = val_probs.max(axis=2)  # shape (num_windows, 250)
print(f"\n{'threshold':>9} | {'frame P':>7} {'frame R':>7} {'frame F1':>8} | "
      f"{'event P':>7} {'event R':>7} {'event F1':>8} {'#pred':>6}")

for threshold in np.arange(0.1, 1.0, 0.1):
    predicted = dog_prob >= threshold
    frames = frame_scores(true_dog, predicted)
    events = event_scores(true_dog, predicted)
    print(f"{threshold:>9.1f} | {frames['precision']:>7.2f} {frames['recall']:>7.2f} "
          f"{frames['f1']:>8.2f} | {events['precision']:>7.2f} {events['recall']:>7.2f} "
          f"{events['f1']:>8.2f} {events['num_predicted']:>6}")
    
print(f"True events in val: {events['num_true']}\n")

for column, sed_class in enumerate(SED_CLASSES):
    truth = val_targets[:, :, column] > 0
    if not truth.any():
        print(f"{sed_class:<12} no true frames in val, skipped")
        continue
    scores = frame_scores(truth, val_probs[:, :, column] >= 0.5)
    print(f"{sed_class:<12} frame P {scores['precision']:.2f}  R {scores['recall']:.2f}  "
          f"F1 {scores['f1']:.2f}")

403 windows of 10 s



C:\Users\kimy8\Desktop\Deep Learning\dog-vocal-interpreter\third_party\PretrainedSED\models\frame_passt\preprocess.py:129: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


100/403 done
200/403 done
300/403 done
400/403 done
403/403 done

Saved to C:\Users\kimy8\Desktop\Deep Learning\dog-vocal-interpreter\data\cache\bark_detector\playpant_val_zero_shot.npz
Frames with a dog sound: 44.7%
bark         0.39% of frames
growl        0.74% of frames
whine        0.15% of frames
howl         0.00% of frames
pant         40.38% of frames
other_vocal  3.18% of frames

threshold | frame P frame R frame F1 | event P event R event F1  #pred
      0.1 |    0.75    0.63     0.69 |    0.70    0.55     0.62   1149
      0.2 |    0.84    0.39     0.53 |    0.84    0.38     0.52   1115
      0.3 |    0.87    0.19     0.31 |    0.86    0.26     0.40    878
      0.4 |    0.90    0.07     0.12 |    0.89    0.15     0.25    456
      0.5 |    0.93    0.02     0.03 |    0.91    0.06     0.11    120
      0.6 |    1.00    0.01     0.01 |    1.00    0.03     0.06     38
      0.7 |    1.00    0.00     0.00 |    1.00    0.01     0.02     17
      0.8 |    0.00    0.00     0.00 | 

These numbers are without any tuning.

### Analysis

As the threshold increases, the precision of the model's prediction also increases (meaning that it rarely raises false alarms, so we can trust its prediction). However, as the threshold increases, the recall of the model's prediction significantly decreases (meaning that many SED classes are missed). 

This is expected because most pants and whine are quiet.

The pretrained SED is good at barks, but almost never fires at pant. Because the playpant dataset contains almost all pant data, the numbers show extremely poor performance for the model's prediction. 